# Neuro-Symbolic Syllogism Validator

A two-stage approach for SemEval 2026 Task 11:
1. **LLM** extracts syllogism structure (NLU task)
2. **Python** applies deterministic validity rules (no LLM reasoning)

This eliminates code generation errors and removes content effect from validity checking.

## 1. Setup and Imports

In [1]:
import sys
import os
import json
from pathlib import Path
from datetime import datetime

_nb_dir = Path.cwd() / "pamaldi_semeval_2026_11_task1" if (Path.cwd() / "pamaldi_semeval_2026_11_task1").exists() else Path.cwd()
sys.path.insert(0, str(_nb_dir))
import setup_path

from load_credentials import load_credentials_from_file
_creds = _nb_dir / "aws_credentials.txt" if (_nb_dir / "aws_credentials.txt").exists() else _nb_dir.parent / "aws_credentials.txt"
load_credentials_from_file(str(_creds))
print('✓ Credentials loaded')

from bedrock_client_bearer import BedrockClientBearer as BedrockClient
from neurosymbolic_pipeline import NeuroSymbolicPipeline
from evaluation import NeuroSymbolicEvaluator
from validity_checker import SyllogismValidityChecker

print('✓ Imports successful')

✓ Credentials loaded
✓ Imports successful


## 2. Configuration

In [2]:
# ============================================================
# CONFIGURATION - Modify these parameters
# ============================================================

# Model configuration - Use Qwen with API Key auth
MODEL_ID = "qwen.qwen3-32b-v1:0"

# Number of samples to process (0 = skip phase, -1 = all)
NUM_TRAIN = 2    # Set to 0 to skip training, -1 for all
NUM_TEST = -1   # Set to 0 to skip test, -1 for all

# Self-consistency configuration (for extraction)
USE_SELF_CONSISTENCY = True  # Set to True to enable self-consistency voting
NUM_CONSISTENCY_SAMPLES = 4   # Number of samples for voting (3-5 recommended)
# Graduated temperature schedule: [0.0, 0.3, 0.5, 0.7, 0.8]
# Sample 1 uses temp=0.0 (deterministic), then increasing exploration
TEMPERATURE_SCHEDULE =  [0.0, 0.3, 0.5, 0.7]

# Figure verification configuration
VERIFY_FIGURE = True  # Set to True to enable figure verification

# LLM Fallback configuration (when extraction fails)
USE_FALLBACK = True           # Set to True to use LLM fallback when extraction fails
FALLBACK_USE_SELF_CONSISTENCY = True  # Set to True for fallback self-consistency
FALLBACK_NUM_SAMPLES = 3      # Number of samples for fallback self-consistency

# Data paths
TRAIN_DATA_PATH = "data/train_data.json"
TEST_DATA_PATH = "data/test_data_subtask_1.json"

# Results directory
RESULTS_DIR = "../semeval_results"

# ============================================================

print(f"Model: {MODEL_ID}")
print(f"NUM_TRAIN: {NUM_TRAIN} {'(skip)' if NUM_TRAIN == 0 else '(all)' if NUM_TRAIN == -1 else ''}")
print(f"NUM_TEST: {NUM_TEST} {'(skip)' if NUM_TEST == 0 else '(all)' if NUM_TEST == -1 else ''}")
print(f"Self-consistency: {USE_SELF_CONSISTENCY}")
if USE_SELF_CONSISTENCY:
    temps = TEMPERATURE_SCHEDULE or [0.0, 0.3, 0.5, 0.7, 0.8]
    print(f"  Samples: {NUM_CONSISTENCY_SAMPLES}, Temperatures: {temps[:NUM_CONSISTENCY_SAMPLES]}")
print(f"Figure verification: {VERIFY_FIGURE}")
print(f"LLM Fallback: {USE_FALLBACK}")
if USE_FALLBACK:
    print(f"  Fallback self-consistency: {FALLBACK_USE_SELF_CONSISTENCY}")

Model: qwen.qwen3-32b-v1:0
NUM_TRAIN: 2 
NUM_TEST: -1 (all)
Self-consistency: True
  Samples: 4, Temperatures: [0.0, 0.3, 0.5, 0.7]
Figure verification: True
LLM Fallback: True
  Fallback self-consistency: True


## 3. Initialize Components

In [3]:
# Initialize Bedrock client
client = BedrockClient(model_id=MODEL_ID)

# Initialize evaluator
evaluator = NeuroSymbolicEvaluator()

print("✓ Components initialized")

✓ Components initialized


## 4. Training Phase

In [4]:
train_results = None
train_metrics = None

if NUM_TRAIN == 0:
    print("⏭ Skipping training phase (NUM_TRAIN = 0)")
else:
    # Load training data
    with open(TRAIN_DATA_PATH, 'r', encoding='utf-8') as f:
        train_data = json.load(f)
    
    # Limit samples if specified
    if NUM_TRAIN > 0:
        train_data = train_data[:NUM_TRAIN]
    
    print(f"Loaded {len(train_data)} training examples")
    
    # Initialize pipeline for training
    train_pipeline = NeuroSymbolicPipeline(
        bedrock_client=client,
        use_reflexion=True,
        max_reflexion_attempts=5,
        use_self_consistency=USE_SELF_CONSISTENCY,
        num_consistency_samples=NUM_CONSISTENCY_SAMPLES,
        temperature_schedule=TEMPERATURE_SCHEDULE,
        verify_figure=VERIFY_FIGURE,
        use_fallback=USE_FALLBACK,
        fallback_use_self_consistency=FALLBACK_USE_SELF_CONSISTENCY,
        fallback_num_samples=FALLBACK_NUM_SAMPLES,
        results_dir=RESULTS_DIR,
        run_name="train"
    )
    
    # Process training data
    train_results = train_pipeline.run(train_data, verbose=True)
    
    # Evaluate
    train_metrics = evaluator.evaluate(train_results)
    evaluator.print_report(train_metrics)

Loaded 2 training examples
  System prompt saved to: ../semeval_results\neurosymbolic_train_20260301_135357\system_prompt.txt
  Results dir: ../semeval_results\neurosymbolic_train_20260301_135357
✓ NeuroSymbolic Pipeline initialized
  Extractor: Original (LLM figure)
  Use reflexion: True
  Use self-consistency: True
    Samples: 4, Temperatures: [0.0, 0.3, 0.5, 0.7]
  Verify figure: True
  Use fallback: True
    Fallback self-consistency: True
    Fallback samples: 3
  Results dir: ../semeval_results\neurosymbolic_train_20260301_135357


Processing: 100%|██████████| 2/2 [00:18<00:00,  9.19s/it]


✓ Summary saved to: ../semeval_results\neurosymbolic_train_20260301_135357
  Methods: 2 symbolic, 0 fallback, 0 failed
  Accuracy: 100.00%

NEURO-SYMBOLIC EVALUATION REPORT

Total Items: 2
Processed: 2
Extraction Rate: 100.0%

Overall Accuracy: 100.00%
  Correct: 2
  Incorrect: 0

By Ground Truth:
  VALID accuracy: 100.00% (n=1)
  INVALID accuracy: 100.00% (n=1)



## 5. Analyze Training Errors (if training was run)

In [5]:
if train_results:
    # Find incorrect predictions
    errors = [r for r in train_results 
              if r.get('ground_truth') and r['prediction'] != r['ground_truth']]
    
    print(f"Total errors: {len(errors)}")
    
    if errors:
        print("\nSample errors:")
        for i, err in enumerate(errors[:3]):
            print(f"\n--- Error {i+1} ---")
            print(f"Text: {err['text'][:100]}...")
            print(f"Ground Truth: {err['ground_truth']}")
            print(f"Prediction: {err['prediction']}")
            if err.get('validity_details'):
                print(f"Form: {err['validity_details'].get('form')}")
                print(f"Reason: {err['validity_details'].get('reason')}")
else:
    print("No training results to analyze")

Total errors: 0


## 6. Test Phase

In [6]:
test_results = None

if NUM_TEST == 0:
    print("⏭ Skipping test phase (NUM_TEST = 0)")
else:
    # Load test data
    with open(TEST_DATA_PATH, 'r', encoding='utf-8') as f:
        test_data = json.load(f)
    
    # Limit samples if specified
    if NUM_TEST > 0:
        test_data = test_data[:NUM_TEST]
    
    print(f"Loaded {len(test_data)} test examples")
    
    # Initialize pipeline for test
    test_pipeline = NeuroSymbolicPipeline(
        bedrock_client=client,
        use_reflexion=True,
        max_reflexion_attempts=5,
        use_self_consistency=USE_SELF_CONSISTENCY,
        num_consistency_samples=NUM_CONSISTENCY_SAMPLES,
        temperature_schedule=TEMPERATURE_SCHEDULE,
        verify_figure=VERIFY_FIGURE,
        use_fallback=USE_FALLBACK,
        fallback_use_self_consistency=FALLBACK_USE_SELF_CONSISTENCY,
        fallback_num_samples=FALLBACK_NUM_SAMPLES,
        results_dir=RESULTS_DIR,
        run_name="test"
    )
    
    # Process test data
    test_results = test_pipeline.run(test_data, verbose=True)
    
    # Save predictions
    test_pipeline.save_predictions(test_results)
    print("\n✓ Predictions saved!")

    # Official evaluation on test (Task 1 & 3)
    print("\n" + "=" * 80)
    print("OFFICIAL EVALUATION — TEST (Task 1 & 3)")
    print("=" * 80)
    test_output_dir = test_pipeline.results_dir
    reference_file = os.path.join(test_output_dir, 'reference.json')
    predictions_file = os.path.join(test_output_dir, 'predictions_official.json')
    results_file = os.path.join(test_output_dir, 'official_evaluation_results.json')
    print("\n[1] Building reference file...")
    reference_data = [{'id': item['id'], 'validity': item.get('validity', False), 'plausibility': item.get('plausibility', False)} for item in test_data]
    with open(reference_file, 'w', encoding='utf-8') as f:
        json.dump(reference_data, f, indent=2)
    print(f"✓ Reference file: {reference_file}")
    print("\n[2] Building predictions file...")
    predictions_official = [{'id': r['id'], 'validity': (r.get('prediction') == 'VALID')} for r in test_results]
    with open(predictions_file, 'w', encoding='utf-8') as f:
        json.dump(predictions_official, f, indent=2)
    print(f"✓ Predictions file: {predictions_file}")
    print("\n[3] Running official evaluation...")
    print("-" * 80)
    _eval_kit = Path.cwd().parent / "evaluation_kit" / "task 1 & 3"
    if _eval_kit.exists() and str(_eval_kit) not in sys.path:
        sys.path.insert(0, str(_eval_kit))
    from evaluation_script import run_full_scoring
    run_full_scoring(reference_file, predictions_file, results_file)
    if os.path.exists(results_file):
        print("\n" + "=" * 80)
        print("OFFICIAL EVALUATION RESULTS (TEST)")
        print("=" * 80)
        with open(results_file, 'r') as f:
            eval_results = json.load(f)
        print(f"\nModel: {MODEL_ID}")
        print(f"Examples: {len(test_data)}")
        print(f"\nMetrics:")
        print(f"  Accuracy:        {eval_results.get('accuracy', 0):.2f}%")
        print(f"  Content Effect:  {eval_results.get('content_effect', 0):.2f}%")
        print(f"  Combined Score:  {eval_results.get('combined_score', 0):.2f}%")
        print("\n" + "=" * 80)

Loaded 191 test examples
  System prompt saved to: ../semeval_results\neurosymbolic_test_20260301_135416\system_prompt.txt
  Results dir: ../semeval_results\neurosymbolic_test_20260301_135416
✓ NeuroSymbolic Pipeline initialized
  Extractor: Original (LLM figure)
  Use reflexion: True
  Use self-consistency: True
    Samples: 4, Temperatures: [0.0, 0.3, 0.5, 0.7]
  Verify figure: True
  Use fallback: True
    Fallback self-consistency: True
    Fallback samples: 3
  Results dir: ../semeval_results\neurosymbolic_test_20260301_135416


Processing: 100%|██████████| 191/191 [39:24<00:00, 12.38s/it] 


✓ Summary saved to: ../semeval_results\neurosymbolic_test_20260301_135416
  Methods: 177 symbolic, 14 fallback, 0 failed

  ┌─────────────────────────────────────┐
  │  COMBINED SCORE:    45.74           │
  │  Accuracy:          97.91%          │
  │  Content Effect:     2.13           │
  └─────────────────────────────────────┘

  Accuracy breakdown: Symbolic 98.31%, Fallback 92.86%

  Failures logged: 4 (see debug_failures.txt)
✓ Predictions saved to: ../semeval_results\neurosymbolic_test_20260301_135416\predictions.json (official format: validity=boolean)

✓ Predictions saved!

OFFICIAL EVALUATION — TEST (Task 1 & 3)

[1] Building reference file...
✓ Reference file: ../semeval_results\neurosymbolic_test_20260301_135416\reference.json

[2] Building predictions file...
✓ Predictions file: ../semeval_results\neurosymbolic_test_20260301_135416\predictions_official.json

[3] Running official evaluation...
--------------------------------------------------------------------------------
